## Where the socioeconomic model works

Notebooks 4 and 5 left a four-term profile for Kaiser share across offered counties, and flagged counties whose signed departures are large. This notebook asks a different question: where does the profile land close to actual share, and what do those counties have in common?

1. Rank offered counties by absolute residual (how far actual share sits from the profile).
2. Slice absolute error by population, uninsured rate, unemployment, benchmark premium, income, poverty, carrier count, footprint, and Northern vs Southern California.
3. Separate reliability (|error|) from direction (over/under), so a large shortfall county is not confused with a county the model simply cannot read.
4. State where a socioeconomic / market score is a useful screening tool for the Kaiser audience, and where it is not.

This is pattern-finding on 31 counties. It is not a formal interaction test.


In [ ]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
})

FOUR_FORMULA = (
    "kp_market_share ~ log_population + uninsured_rate + unemployment_rate "
    "+ benchmark_silver_premium"
)
SOUTHERN = ["Imperial", "Kern", "Orange", "Riverside", "San Bernardino", "San Diego", "Ventura"]
from pathlib import Path
FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def short_county(name):
    return name.replace(" County", "")


con = duckdb.connect("../data/processed/analysis.duckdb", read_only=True)
market = con.sql("select * from main_marts.mart_cty_yr_market").df()
performance = con.sql("select * from main_marts.mart_cty_yr_performance").df()
socioeconomic = con.sql("select * from main_marts.mart_cty_yr_socioeconomic").df()
footprint = con.sql("""
    select
        year,
        county_fips,
        bool_and(coalesce(is_partial_county, false)) as partial_county
    from main_staging.stg_cms_service_areas
    where issuer_id = '40513'
      and service_area_id = 'CAS002'
    group by 1, 2
""").df()

panel = (
    market.merge(performance, on=["county_fips", "year"], how="left", suffixes=("", "_performance"))
    .merge(socioeconomic, on=["county_fips", "year"], how="left", suffixes=("", "_socio"))
    .merge(footprint, on=["county_fips", "year"], how="left")
)
panel["county"] = panel["county_name"].map(short_county)
panel["log_population"] = np.log(panel["population_total"])
panel["income_10k"] = panel["median_household_income"] / 10_000

offered = panel[panel["kp_offered"] & panel["year"].between(2016, 2024)].copy()
measure_cols = [
    "kp_market_share",
    "population_total",
    "median_household_income",
    "income_10k",
    "poverty_rate",
    "uninsured_rate",
    "unemployment_rate",
    "carrier_count",
    "benchmark_silver_premium",
]
county = offered.groupby(["county_fips", "county"], as_index=False)[measure_cols].mean()
county["log_population"] = np.log(county["population_total"])
sample = county.dropna(subset=["kp_market_share", "log_population", "uninsured_rate", "unemployment_rate", "benchmark_silver_premium"]).copy()
sample["southern"] = sample["county"].isin(SOUTHERN).astype(int)

profile_model = smf.ols(FOUR_FORMULA, data=sample).fit(cov_type="HC1")
regional_model = smf.ols(FOUR_FORMULA + " + southern", data=sample).fit(cov_type="HC1")
residual_sd = float(np.sqrt(profile_model.mse_resid))
regional_sd = float(np.sqrt(regional_model.mse_resid))

display(Markdown(
    f"The four-term profile is refit on **{len(sample)} counties** "
    f"(2016–2024 averages; Los Angeles and Monterey are out). "
    f"Adjusted R² is **{profile_model.rsquared_adj:.2f}**; residual SE is "
    f"**{residual_sd * 100:.1f}** percentage points. "
    f"With the Southern California indicator, adjusted R² is **{regional_model.rsquared_adj:.2f}** "
    f"and residual SE is **{regional_sd * 100:.1f}** points."
))


## Absolute error and signed departure

Reliability is absolute residual `|e|`: how far the profile sits from actual share, regardless of direction. Direction is the signed departure from notebook 5. Counties are grouped into absolute-error tertiles (low / mid / high). A second cut uses fractions of the residual standard error.


In [ ]:
errors = sample[[
    "county_fips", "county", "kp_market_share", "population_total", "log_population",
    "median_household_income", "income_10k", "poverty_rate", "uninsured_rate",
    "unemployment_rate", "carrier_count", "benchmark_silver_premium", "southern",
]].copy()
errors["expected_share"] = profile_model.fittedvalues.to_numpy()
errors["departure"] = profile_model.resid.to_numpy()
errors["departure_pp"] = errors["departure"] * 100
errors["abs_error_pp"] = errors["departure_pp"].abs()
errors["departure_with_region_pp"] = regional_model.resid.to_numpy() * 100
errors["abs_error_with_region_pp"] = errors["departure_with_region_pp"].abs()

year_2025 = panel.loc[panel["year"] == 2025, [
    "county_fips", "partial_county", "kp_market_share", "total_marketplace_enrollment",
]].rename(columns={"kp_market_share": "share_2025", "total_marketplace_enrollment": "market_2025"})
errors = errors.merge(year_2025, on="county_fips", how="left")
errors["footprint"] = np.where(errors["partial_county"], "Some ZIP codes", "Whole county")
errors["region"] = np.where(errors["southern"].eq(1), "Southern California", "Northern California")
errors["error_tert"] = pd.qcut(
    errors["abs_error_pp"],
    q=3,
    labels=["Low |error|", "Mid |error|", "High |error|"],
)
errors["error_band"] = pd.cut(
    errors["abs_error_pp"],
    bins=[-np.inf, 0.5 * residual_sd * 100, residual_sd * 100, np.inf],
    labels=["Within 0.5 SE", "0.5–1 SE", "Beyond 1 SE"],
)
errors["direction"] = np.select(
    [errors["departure_pp"] <= -residual_sd * 100, errors["departure_pp"] >= residual_sd * 100],
    ["Below profile", "Above profile"],
    default="Near profile",
)
errors = errors.sort_values("abs_error_pp").reset_index(drop=True)

county_view = pd.DataFrame({
    "county": errors["county"],
    "error_tert": errors["error_tert"],
    "error_band": errors["error_band"],
    "direction": errors["direction"],
    "abs_error_pp": errors["abs_error_pp"].map(lambda v: f"{v:.1f}"),
    "departure_pp": errors["departure_pp"].map(lambda v: f"{v:+.1f}"),
    "with_region_pp": errors["departure_with_region_pp"].map(lambda v: f"{v:+.1f}"),
    "share": (errors["kp_market_share"] * 100).map(lambda v: f"{v:.0f}%"),
    "expected": (errors["expected_share"] * 100).map(lambda v: f"{v:.0f}%"),
    "region": errors["region"],
    "footprint": errors["footprint"],
    "market_2025": errors["market_2025"].map(lambda v: f"{v:,.0f}"),
})
display(HTML(county_view.to_html(index=False)))

low = errors[errors["error_tert"].eq("Low |error|")]
high = errors[errors["error_tert"].eq("High |error|")]
display(Markdown(
    f"Low absolute-error counties (best fit): **{', '.join(low['county'])}**. "
    f"High absolute-error counties: **{', '.join(high['county'])}**. "
    f"Median `|error|` is **{errors['abs_error_pp'].median():.1f}** points; "
    f"**{(errors['error_band'].eq('Within 0.5 SE')).sum()}** counties sit within half a residual SE "
    f"({0.5 * residual_sd * 100:.1f} pp) and "
    f"**{(errors['error_band'].eq('Beyond 1 SE')).sum()}** sit beyond one SE "
    f"({residual_sd * 100:.1f} pp)."
))


## What is common in well-fit and poorly-fit counties?

Each continuous condition is summarized by absolute-error tertile (median). Categorical shares show Northern vs Southern California and whole-county vs ZIP-limited footprint. Spearman correlations link `|error|` to each continuous measure across all 31 counties.


In [ ]:
SLICE_VARS = [
    ("population_total", "Population", "{:,.0f}"),
    ("uninsured_rate", "Uninsured rate", "{:.1f}%"),
    ("unemployment_rate", "Unemployment rate", "{:.1f}%"),
    ("benchmark_silver_premium", "Benchmark Silver premium", "${:,.0f}"),
    ("median_household_income", "Median household income", "${:,.0f}"),
    ("poverty_rate", "Poverty rate", "{:.1f}%"),
    ("carrier_count", "Carrier count", "{:.1f}"),
    ("market_2025", "2025 marketplace enrollment", "{:,.0f}"),
    ("kp_market_share", "Kaiser share (2016–2024 avg)", "{:.0%}"),
]

tert_order = ["Low |error|", "Mid |error|", "High |error|"]
summary_rows = []
for col, label, fmt in SLICE_VARS:
    medians = errors.groupby("error_tert", observed=True)[col].median()
    row = {"measure": label}
    for tert in tert_order:
        row[tert] = fmt.format(medians.loc[tert])
    summary_rows.append(row)
slice_table = pd.DataFrame(summary_rows)
display(Markdown("**Median conditions by absolute-error tertile**"))
display(HTML(slice_table.to_html(index=False)))

cat_rows = []
for col, label in [("region", "Region"), ("footprint", "Footprint")]:
    for tert in tert_order:
        counts = errors.loc[errors["error_tert"].eq(tert), col].value_counts(normalize=True)
        cat_rows.append({
            "slice": label,
            "error_tert": tert,
            **{str(k): f"{v:.0%}" for k, v in counts.items()},
        })
cat_table = pd.DataFrame(cat_rows).fillna("0%")
display(Markdown("**Share of counties in each category, by absolute-error tertile**"))
display(HTML(cat_table.to_html(index=False)))

corr_rows = []
for col, label, _ in SLICE_VARS:
    rho = errors["abs_error_pp"].corr(errors[col], method="spearman")
    corr_rows.append({"measure": label, "spearman_with_|error|": f"{rho:+.2f}"})
# also with southern indicator
rho_s = errors["abs_error_pp"].corr(errors["southern"], method="spearman")
corr_rows.append({"measure": "Southern California (0/1)", "spearman_with_|error|": f"{rho_s:+.2f}"})
corr_table = pd.DataFrame(corr_rows)
display(Markdown("**Spearman correlation of each measure with absolute residual**"))
display(HTML(corr_table.to_html(index=False)))

# narrative helpers
def med(frame, col):
    return frame[col].median()

display(Markdown(
    f"Low-error counties have median population **{med(low, 'population_total'):,.0f}**, "
    f"uninsured **{med(low, 'uninsured_rate'):.1f}%**, unemployment **{med(low, 'unemployment_rate'):.1f}%**, "
    f"and benchmark premium **${med(low, 'benchmark_silver_premium'):,.0f}**. "
    f"High-error counties sit at **{med(high, 'population_total'):,.0f}**, "
    f"**{med(high, 'uninsured_rate'):.1f}%**, **{med(high, 'unemployment_rate'):.1f}%**, "
    f"and **${med(high, 'benchmark_silver_premium'):,.0f}**. "
    f"Southern California is **{(low['region'].eq('Southern California')).mean():.0%}** of the low-error group "
    f"and **{(high['region'].eq('Southern California')).mean():.0%}** of the high-error group. "
    f"ZIP-limited footprint is **{(low['footprint'].eq('Some ZIP codes')).mean():.0%}** vs "
    f"**{(high['footprint'].eq('Some ZIP codes')).mean():.0%}**."
))


## Error across predictor bins

For each profile term (and a few descriptive extras), counties are split at the median into lower and higher halves. Mean absolute residual in each half shows whether the profile is systematically less reliable on one side of the distribution.


In [ ]:
BIN_VARS = [
    ("log_population", "Population (log)"),
    ("uninsured_rate", "Uninsured rate"),
    ("unemployment_rate", "Unemployment rate"),
    ("benchmark_silver_premium", "Benchmark Silver premium"),
    ("median_household_income", "Median household income"),
    ("poverty_rate", "Poverty rate"),
    ("carrier_count", "Carrier count"),
]

bin_rows = []
bin_plot = []
for col, label in BIN_VARS:
    median = errors[col].median()
    lower = errors[errors[col] <= median]
    higher = errors[errors[col] > median]
    low_mae = lower["abs_error_pp"].mean()
    high_mae = higher["abs_error_pp"].mean()
    bin_rows.append({
        "measure": label,
        "lower_half_mean_|e|": f"{low_mae:.1f}",
        "upper_half_mean_|e|": f"{high_mae:.1f}",
        "gap_pp": f"{high_mae - low_mae:+.1f}",
        "n_lower": len(lower),
        "n_upper": len(higher),
    })
    bin_plot.append({"measure": label, "half": "At or below median", "mae": low_mae})
    bin_plot.append({"measure": label, "half": "Above median", "mae": high_mae})

display(HTML(pd.DataFrame(bin_rows).to_html(index=False)))

plot_df = pd.DataFrame(bin_plot)
fig, ax = plt.subplots(figsize=(9, 5))
measures = [label for _, label in BIN_VARS]
y = np.arange(len(measures))
width = 0.38
low_vals = [plot_df[(plot_df.measure == m) & (plot_df.half == "At or below median")].mae.iloc[0] for m in measures]
high_vals = [plot_df[(plot_df.measure == m) & (plot_df.half == "Above median")].mae.iloc[0] for m in measures]
ax.barh(y - width / 2, low_vals, height=width, color="#98a2b3", label="At or below median")
ax.barh(y + width / 2, high_vals, height=width, color="#1f4e79", label="Above median")
ax.set_yticks(y)
ax.set_yticklabels(measures)
ax.set_xlabel("Mean absolute residual (percentage points)")
ax.set_title("Profile absolute error by predictor half")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_abs_error_by_predictor_half.png", dpi=150, bbox_inches="tight")
plt.show()


## Map of absolute error

The map is absolute residual in percentage points for counties in the profile fit. Hatching marks a ZIP-limited 2025 service area. Los Angeles and Monterey are offered but outside the fit.


In [ ]:
import shapefile
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch, Polygon

boundaries = shapefile.Reader("../data/raw/census/boundaries/cb_2023_us_county_5m.zip")
X_SCALE = np.cos(np.radians(37.5))
county_rings = {}
for item in boundaries.iterShapeRecords():
    if item.record["STATEFP"] != "06":
        continue
    points = np.array(item.shape.points)
    points[:, 0] *= X_SCALE
    breaks = list(item.shape.parts) + [len(points)]
    county_rings[item.record["GEOID"]] = [points[start:end] for start, end in zip(breaks[:-1], breaks[1:])]

NOT_OFFERED = "#f2f4f7"
OUTSIDE_FIT = "#d0d5dd"
abs_colors = LinearSegmentedColormap.from_list("abs_error", ["#edf2f7", "#1f4e79"])
map_2025 = panel.loc[panel["year"] == 2025, ["county_fips", "kp_offered", "partial_county"]]
offered_2025 = map_2025[map_2025["kp_offered"]]
abs_norm = Normalize(0, np.ceil(errors["abs_error_pp"].max() / 5) * 5)
abs_fill = {fips: OUTSIDE_FIT for fips in offered_2025["county_fips"]}
abs_fill.update({
    row.county_fips: abs_colors(abs_norm(row.abs_error_pp)) for row in errors.itertuples()
})
zip_limited = set(offered_2025.loc[offered_2025["partial_county"].fillna(False).astype(bool), "county_fips"])


def draw_counties(ax, fills, hatched=()):
    for fips, rings in county_rings.items():
        for ring in rings:
            ax.add_patch(Polygon(
                ring,
                closed=True,
                facecolor=fills.get(fips, NOT_OFFERED),
                edgecolor="white",
                linewidth=0.6,
                hatch="////" if fips in hatched else None,
                hatchcolor="#667085",
            ))
    ax.set_aspect("equal")
    ax.autoscale_view()
    ax.axis("off")


fig, ax = plt.subplots(figsize=(8, 9))
draw_counties(ax, abs_fill, hatched=zip_limited)
for row in errors[errors["error_tert"].eq("High |error|")].itertuples():
    ring = max(county_rings[row.county_fips], key=len)
    label = f"{row.county}\n{row.abs_error_pp:.0f}"
    ax.text(ring[:, 0].mean(), ring[:, 1].mean(), label,
            ha="center", va="center", fontsize=7, color="#101828")
sm = ScalarMappable(norm=abs_norm, cmap=abs_colors)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("|Actual − profile| (percentage points)")
ax.set_title("Absolute residual from the four-term profile")
ax.legend(
    handles=[
        Patch(facecolor=OUTSIDE_FIT, edgecolor="white", label="Offered, outside fit"),
        Patch(facecolor=NOT_OFFERED, edgecolor="white", label="Not offered"),
        Patch(facecolor="white", edgecolor="#667085", hatch="////", label="ZIP-limited footprint"),
    ],
    loc="lower left",
    frameon=False,
    fontsize=8,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_abs_error_map.png", dpi=150, bbox_inches="tight")
plt.show()


## Reliability vs direction

High absolute error includes both shortfalls and outperformers from notebook 5. The useful distinction for a Kaiser audience is whether the score is close enough to trust as a screen, not only whether the county is above or below the profile.


In [ ]:
cross = (
    errors.groupby(["error_tert", "direction"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=tert_order)
)
display(Markdown("**Count of counties by absolute-error tertile and signed direction**"))
display(HTML(cross.to_html()))

near = errors[errors["error_band"].eq("Within 0.5 SE")]
beyond = errors[errors["error_band"].eq("Beyond 1 SE")]
display(Markdown(
    f"Counties within 0.5 SE ({len(near)}): **{', '.join(near['county'])}**. "
    f"Counties beyond 1 SE ({len(beyond)}): **{', '.join(beyond['county'])}**. "
    f"Among beyond-1-SE counties, "
    f"**{(beyond['direction'].eq('Below profile')).sum()}** are shortfalls and "
    f"**{(beyond['direction'].eq('Above profile')).sum()}** are outperformers."
))

# confidence tag for later forecast notebook reuse
errors["confidence"] = np.select(
    [
        errors["error_band"].eq("Within 0.5 SE"),
        errors["error_band"].eq("0.5–1 SE"),
    ],
    ["Higher", "Moderate"],
    default="Lower",
)
# Southern / opening caveats bump caution in narrative even if abs error is mid
confidence_view = errors[[
    "county", "confidence", "abs_error_pp", "departure_pp", "region", "footprint", "market_2025"
]].sort_values(["confidence", "abs_error_pp"])
display(Markdown("**Screening-confidence tags** (for use with a 2027 score): Higher = within 0.5 residual SE historically."))
display(HTML(pd.DataFrame({
    "county": confidence_view["county"],
    "confidence": confidence_view["confidence"],
    "abs_error_pp": confidence_view["abs_error_pp"].map(lambda v: f"{v:.1f}"),
    "departure_pp": confidence_view["departure_pp"].map(lambda v: f"{v:+.1f}"),
    "region": confidence_view["region"],
    "footprint": confidence_view["footprint"],
}).to_html(index=False)))


## What this tells the Kaiser audience

**Where the tool helps.** The four-term socioeconomic / market profile is a between-county screening score. It is most useful where absolute historical error is low: the profile's reading of share is close to what Kaiser actually holds, so ranking or comparing those counties on conditions is less likely to mislead. Within Northern California, and where unemployment and uninsured rates sit nearer the footprint's better-performing pattern, the score is a reasonable first pass before ops or network review.

**Where the tool is weak.** Large absolute residuals concentrate the shortfalls and outperformers from notebook 5. Southern California counties sit about 19 points below Northern peers with the same conditions once region is added; a statewide SES score without that context overstates what conditions alone imply in the South. ZIP-limited footprints and large markets can also make a county-level share hard to read. Within a county over time, only about 3% of year-adjusted share variation moves with these conditions, so the same model is a poor lever for year-to-year change.

**Decision rule.** Use the profile to screen which counties' conditions support higher share, and attach the absolute-error / confidence tag before acting. Do not treat a high fitted share as a forecast of what Kaiser will achieve without checking the signed departure, region, and footprint. Notebook 7 carries those departures forward when scoring 2027 conditions.
